In [ ]:
import math
import torch
import random
import numpy as np
from torch import nn, optim

from datetime import datetime
from dataclasses import dataclass, asdict
from tokenizer import TinyStoriesTokenizer
from self_attention import MultiHeadSelfAttention

## Transformer block
To obtain a complete transformer block, we add a __position-wise feed-forward layer__ after the self-attention computation. The vectors are first projected to be 4 times wider, a non-linear ``relu'' function is applied, and the vectors are projected down to their original size.

Before and after the self-attention computation, we add a __layer norm__ which normalizes each vector to have mean 0 and variance 1. This contributes to "numerical stability" in the forward pass, i.e. that the numbers within the vectors stay within the bounds representable by a computer. 

We also add __residual connections__ to each layer. In general, this means that we compute $x+f(x)$ rather than just $f(x)$ (the "$x+$" is sometimes called the "identity shortcut"). The gradient of $x+f(x)$ is $1+f'(x)$, which is close to 1 if $f'(x)$ is tiny. Without the identity shortcut, the overall gradient in the backward pass could be come extremely small, in particular in a deep network where many layers are stacked. The residual connection thus acts as a prevention for the "vanishing gradient" problem.



In [ ]:
class PositionwiseFFN(nn.Module):
    """
    The position-wise FFN that follows after the self-attention computation.
    Vectors are projected to 4x the dimensionality and then projected down
    again after relu application.
    """

    def __init__(self, vector_dim, dropout_prob) :
        super().__init__()
        self.fc1 = nn.Linear(vector_dim, 4*vector_dim, bias=True)
        self.fc2 = nn.Linear(4*vector_dim, vector_dim, bias=True)
        self.dropout = nn.Dropout(dropout_prob)

    def forward(self, x):
        return self.fc2(self.dropout(torch.relu(self.fc1(x))))

class Block(nn.Module):
    """
    Transformer encoder block.

    This version differs from the original version in  [Vaswani et al. NeurIPS 2017],
    and applies the LayerNorm before the self-attention, and before the FFN, as this
    has proved to be beneficial (see [Nguyen and Salazar 2019]).
    """

    def __init__(self, vector_dim, n_heads, block_size, dropout_prob):
        super().__init__()
        att_dim = vector_dim // n_heads
        self.attn = MultiHeadSelfAttention(vector_dim, n_heads, block_size, is_causal=True)
        self.ffn = PositionwiseFFN(vector_dim, dropout_prob)
        self.dropout = nn.Dropout(dropout_prob)
        self.ln1 = nn.LayerNorm(vector_dim)
        self.ln2 = nn.LayerNorm(vector_dim)

    def forward(self, x):
        x1 = self.ln1(x)
        x2 = x + self.dropout(self.attn(x1))
        x3 = self.ln2(x2)
        x4 = x2 + self.dropout(self.ffn(x3))
        return x4

## A generative language model
Next, we will define a generative language model suitable for the "Tiny Stories" dataset. Our goal is to train it to generate short stories which are similar to the ones in the dataset, without exactly repeating any story in the dataset.

The model will have 4 transformer blocks, each with 4 attention heads. The vector size will be 256, and the context window can contain up to 512 tokens (which is the approximate length of a story in the dataset). 

In the forward function, you should __look up the embeddings__ of all the input tokens, __add positional embeddings__, and __apply all the transformer blocks__ in a loop. Finally, you should construct the logits by __applying the final layer__ to the result of the transformer computations. Your task is to implement this function.

In [ ]:
# ============= Hyper-parameters for training ============== #

@dataclass
class Config :
    vocab_size: int = 5000  # This number should agree with the tokenizer
    number_of_transformer_blocks: int = 4
    number_of_attention_heads: int = 4
    vector_dim: int = 256
    block_size: int = 512
    dropout_prob: float = 0.1
    batch_size: int = 8
    learning_rate: float = 0.0005
    weight_decay: float = 0.000001
    no_of_epochs: int = 1


class TinyStoriesLM(nn.Module):

    def __init__(self, config):
        super(TinyStoriesLM, self).__init__()
        self.config = config
        self.embed =  nn.Embedding(config.vocab_size, config.vector_dim)
        self.positional = nn.Parameter(torch.randn(1, config.block_size, config.vector_dim))
        modules = [Block(config.vector_dim,\
                         config.number_of_attention_heads,\
                         config.block_size,\
                         config.dropout_prob) for _ in range(config.number_of_transformer_blocks)]
        self.transformers = nn.ModuleList(modules)
        self.final = nn.Linear(config.vector_dim, config.vocab_size)

    def forward(self, x):

        # YOUR CODE HERE
        
        return logits

    def generate(self, ids, max_new_tokens, temperature=1.0, top_k=None):
        self.eval()
        for _ in range(max_new_tokens):
            ids = ids[-self.config.block_size:]  # Make sure that we don't input more that `block_size` tokens
            logits = self.forward(ids.unsqueeze(0)).squeeze(0)
            logits = logits[-1, :]  # We only want the last time step
            logits = logits / temperature
            if top_k is not None:
                # Only consider the top k classes 
                top_idx = torch.topk(logits, top_k)[1]
                mask = torch.ones(self.config.vocab_size, dtype=torch.bool)
                mask[top_idx] = False
                logits[mask] = float('-inf')
            probs = torch.softmax(logits, dim=-1)
            # Sample from the distribution and append the new token to the input
            next_id = torch.multinomial(probs, num_samples=1)
            ids = torch.cat((ids, next_id), dim=0)
        return ids
        
    @classmethod
    def load(cls, checkpoint_path, device='cpu'):
        """
        Loads a model from a checkpoint file.
        Automatically reconstructs the config and model architecture.
        """
        checkpoint = torch.load(checkpoint_path, map_location=device)
        config = Config(**checkpoint['config'])
        model = cls(config)
        model.load_state_dict(checkpoint['model_state_dict'])
        model.to(device)
    
        print(f"Model loaded from {checkpoint_path} (Epoch {checkpoint['epoch']}, iteration {checkpoint['iteration']})")
        return model


### Dataset 
In order to save disk space, we have already tokenized the Tiny Stories dataset for you (this allows every group to share the same file rather than construct theit own, several-GB file). The pretokenized dataset resides in a binary file, and the "TinyStoriesDataset" class below reads from this file. 

The "ResumableRandomSampler" class allows us to resume training after a pause, without repeating any previously processed datapoints from the training set. 

In [ ]:
from torch.utils.data import Sampler, SubsetRandomSampler
from torch.utils.data import Dataset, DataLoader
from tokenizer import TinyStoriesTokenizer

class TinyStoriesDataset(Dataset):
    def __init__(self, data_file, block_size):
        """
        data_file: path to the .bin file (uint16 array of token IDs)
        block_size: the context window (e.g., 256 or 512 tokens)
        """

        # Memory-map the data file (RAM usage stays near zero!)
        self.data = np.memmap(data_file, dtype=np.uint16, mode='r')
        self.block_size = block_size

    def __len__(self):
        # We subtract block_size to ensure we don't go out of bounds
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        # Pull a chunk of length block_size + 1 (data and target)
        chunk = self.data[idx : idx + self.block_size + 1]

        # Convert to torch tensors
        x = torch.from_numpy(chunk[:-1].astype(np.int64)) # Input
        y = torch.from_numpy(chunk[1:].astype(np.int64))  # Target (shifted by 1)

        return x, y 


# This class facilitates re-starting training without repeating any past training examples
class ResumableRandomSampler(Sampler):
    def __init__(self, data_source, start_index=0, seed=42):
        self.data_source = data_source
        self.start_index = start_index
        self.seed = seed
        self.generator = torch.Generator()
        self.generator.manual_seed(self.seed)
        
        # 1. Generate the full shuffled list of indices once
        self.indices = torch.randperm(len(self.data_source), generator=self.generator).tolist()

    def __iter__(self):
        # 2. Slice the indices to only include those from start_index onwards
        return iter(self.indices[self.start_index:])

    def __len__(self):
        return len(self.indices) - self.start_index

### One sequence yields many training examples
A common misconception is that one "sequence" (a chunk of text) equals one "training example." But, as illustrated by the example in the next cell, the sequence contains as many training examples as tokens.

When we provide the model with a sequence of length $T$ (our ``block_size``), the model generates $T$ separate sets of logits (predictions). Because the causal mask prevents the model from "looking ahead," each position $t$ must predict the token at $t+1$ using only the information from indices $0\ldots t$.

In [ ]:
# Let's have a look at some data
training_dataset = TinyStoriesDataset('/datasets/dd2417/train.bin', config.block_size)
tokenizer = TinyStoriesTokenizer.load('/datasets/dd2417/tokenizer.json')
sample_idx = 42  
x_sample, y_sample = training_dataset[sample_idx]

# We'll just look at the first 10 tokens for clarity
tokens_x = x_sample[:10].tolist()
tokens_y = y_sample[:10].tolist()
print(tokens_x)
print(tokens_y)
print()

print(f"{'Context (Input x)':<40} | {'Target (Label y)':<15}")
print("-" * 60)

for i in range(1, len(tokens_x) + 1):
    # The context grows as we move through the sequence
    context_ids = tokens_x[:i]
    context_text = "".join(tokenizer.vocab[id] for id in context_ids)
    
    # The target is always the 'next' token in the sequence
    target_id = tokens_y[i-1]
    target_text = tokenizer.vocab[target_id]
    
    print(f"{context_text:>40} ---> {target_text:<15}")

If we have a batch size of 8 and a sequence length of 512, the model will return a (8, 512, 5000) tensor of logits (where 5000 is the number of token classes), and we have a (8, 512) tensor of targets. During training, we now reshape these into a (8$\times$ 512, 5000) tensor of logits and an (8 $\times$ 512) tensor of targets, respectively. The ``nn.CrossEntropyLoss`` class will compute all the 8$\times$ 512 = 4096 individual losses and average them in a single forward pass. With a GPU, this can be done very efficiently. 

It is this tremendous efficiency that has popularized the Transformer architecture. With older architectures like the Recurrent Neural Network (RNN), the model would have to make 4096 forward passes to achieve the same progress.

In each iteration of the training loop below, we input a sequence of text containing ``block_size`` tokens. It might happen the one story ends and another story starts in the middle of such a sequence, but we don't care too much about this. Validation loss is computed on a separate "dev" set after every 100 training sequences.  The dev set is small (about 1% of the training set). A model checkpoint is saved every 10,000 training sequences. The checkpoint can be used for further training, or for inference (i.e., using the model). 

In [ ]:
import os

# ======================= Training ======================= #

# Remove the checkpoint file if you want to train from scratch
checkpoint_path = 'last_checkpoint.pt'
RESUME_TRAINING = os.path.exists(checkpoint_path) 

seed = 42
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print( "Running on", device )

tokenizer = TinyStoriesTokenizer.load('/datasets/dd2417/tokenizer.json')

if RESUME_TRAINING:
    checkpoint = torch.load(checkpoint_path, map_location=device)
    print(f"Model loaded from {checkpoint_path} (Epoch {checkpoint['epoch']+1}, iteration {checkpoint['iteration']})")
    config = Config(**checkpoint['config'])
    lm = TinyStoriesLM(config)
    # Sanity check
    if tokenizer.vocab_size != lm.config.vocab_size:
        print("WARNING: The tokenizer's and the model's vocab_size are different!")
    lm.load_state_dict(checkpoint['model_state_dict'])
    lm.to(device) 
else :
    config = Config()
    config.vocab_size = tokenizer.vocab_size
    lm = TinyStoriesLM(config).to(device)

lm_optimizer = torch.optim.AdamW(lm.parameters(), lr=lm.config.learning_rate, weight_decay=config.weight_decay)

if RESUME_TRAINING:
    lm_optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

training_dataset = TinyStoriesDataset('/datasets/dd2417/train.bin', config.block_size)

# Development set 
dev_dataset = TinyStoriesDataset('/datasets/dd2417/dev.bin', config.block_size)
# Create indices that jump by the block size. This ensures we only see each 
# chunk of text once when computing validation loss.
dev_indices = list(range(0, len(dev_dataset), config.block_size))
dev_subset = torch.utils.data.Subset(dev_dataset, dev_indices)
dev_loader = DataLoader(dev_subset, config.batch_size, pin_memory=True)

print( "There are", len(training_dataset), "datapoints and", tokenizer.vocab_size, "token classes in the dataset" ) 

criterion = nn.CrossEntropyLoss()

lm.train()
running_loss = 0
print( datetime.now().strftime("%X"), "Training starts" )
# Can we pick up where we left off?
start_epoch = checkpoint['epoch'] if RESUME_TRAINING else 0
start_iter = checkpoint['iteration'] if RESUME_TRAINING else 0
for epoch in range(start_epoch, config.no_of_epochs) :
    iteration = start_iter if epoch == start_epoch else 0
    # Random sampling of datapoints. Note that we use a new seed for each epoch 
    # -- this will guarantee that we can resume the training anywhere without 
    # repeating any datapoints within an epoch.
    sampler = ResumableRandomSampler(training_dataset, 
                                     start_index=iteration * config.batch_size, 
                                     seed=seed+epoch) 
    training_loader = DataLoader(training_dataset, 
                                 batch_size=config.batch_size, 
                                 sampler=sampler, 
                                 num_workers=4)
    for x,y in training_loader:
        x,y = x.to(device), y.to(device)
        lm_optimizer.zero_grad()
        logits = lm(x)
        loss = criterion(logits.reshape(-1, tokenizer.vocab_size), y.reshape(-1))
        loss.backward()
        lm_optimizer.step()
        running_loss += loss.detach().item()
        iteration += 1
        if iteration%100 == 0:
            # Compute validation loss every 100 iterations
            lm.eval()   
            with torch.no_grad():
                total_loss = 0    
                for vx, vy in dev_loader:
                    vx,vy = vx.to(device), vy.to(device)
                    logits = lm(vx)
                    val_loss = criterion(logits.reshape(-1, tokenizer.vocab_size), vy.reshape(-1))
                    total_loss += val_loss.item()
                print(f'{datetime.now().strftime("%X")} Epoch {epoch+1}, iteration {iteration}: loss={running_loss/100:.4f}, dev loss={total_loss/len(dev_loader):.4f}')
            lm.train()
            running_loss = 0
        if iteration%10000 == 0:
            checkpoint = {
                'model_state_dict': lm.state_dict(),
                'optimizer_state_dict': lm_optimizer.state_dict(),
                'epoch': epoch,
                'iteration': iteration,
                'config': config.__dict__, # Useful so you know the architecture later
            }

            # Overwrites the same file every time to save space
            torch.save(checkpoint, checkpoint_path)
            print(f"Checkpoint saved to {checkpoint_path}")

            # Show progress by generating a story
            lm.eval()
            prompt = "Once upon a time"
            _, start_ids = tokenizer.tokenize(prompt)
            x = torch.tensor(start_ids, dtype=torch.long, device=device)
            with torch.no_grad():
                y = lm.generate(x, max_new_tokens=100, temperature=0.8, top_k=40)
            decoded_output = [tokenizer.vocab[i] for i in y.tolist()]
            print("-" * 30)
            print("".join(decoded_output))
            print("-" * 30)
            lm.train()

### Use the model to generate stories
Try varying the prompt, max tokens, temperature, and top-k sampling to study the effects.

In [ ]:
def tell_story(model, tokenizer, prompt="Once upon a time", max_tokens=100):
    device = next(model.parameters()).device
    
    # Encode prompt
    _, start_ids = tokenizer.tokenize(prompt)
    print(start_ids)
    x = torch.tensor(start_ids, dtype=torch.long, device=device)
    
    # Generate
    with torch.no_grad():
        y = model.generate(x, max_new_tokens=max_tokens, temperature=0.8, top_k=40)
    
    # Decode and print
    decoded_output = [tokenizer.vocab[i] for i in y.tolist()]
    print("-" * 30)
    print("".join(decoded_output))
    print("-" * 30)

checkpoint_path = "last_checkpoint.pt"
tokenizer_path = "/datasets/dd2417/tokenizer.json"
model = TinyStoriesLM.load(checkpoint_path, device='cuda')
tokenizer = TinyStoriesTokenizer.load(tokenizer_path)
tell_story(model, tokenizer)
